
---

## 📚 Sobre este Material

Este material ha sido diseñado con el propósito de **capacitar, actualizar y practicar** conceptos fundamentales de Markdown en Jupyter Notebook. Es una herramienta pensada para facilitar el aprendizaje y la documentación efectiva de proyectos de análisis de datos y ciencia de datos.

### 🤝 Compartir y Colaborar

Este contenido es **libre para compartir, revisar, divulgar y mejorar**. Se promueve activamente su distribución en la comunidad para que más personas puedan beneficiarse y contribuir a su mejora continua. Tu feedback y sugerencias son siempre bienvenidos.

### 👨‍💻 Autor

**Andrés Muñoz**  
*AI & Data Strategy Leader passionate about NLP, LLMs, and MLOps. Driving innovation with data*

- 💼 LinkedIn: [in/amms1989](https://linkedin.com/in/amms1989)
- 🐙 GitHub: [https://github.com/anguihero](https://github.com/anguihero)

---

# Sesión 10: Ensambles y Benchmarking de Modelos

**Autor:** anmmunozsa@outlook.es · Material de código abierto para compartir y aprender colectivamente.

## 🎯 Objetivo de la sesión
Entender los métodos de ensamble (Random Forest, Gradient Boosting, XGBoost) y aprender a comparar objetivamente varios modelos con validación cruzada.

## 🗺️ Tabla de Contenido
1. [Introducción: ¿por qué combinar modelos?](#intro)
2. [Recuperando el dataset y modelos de la Sesión 09](#dataset)
3. [Random Forest (Bagging)](#rf)
4. [Gradient Boosting](#gb)
5. [XGBoost](#xgb)
6. [Validación cruzada k-fold](#cv)
7. [Benchmarking: la tabla comparativa final](#benchmark)
8. [Ejemplos de aplicación real](#aplicaciones)
9. [Retos de práctica](#retos)


<a id="intro"></a>
## 1. Introducción (para dummies)

Un solo árbol de decisión es inestable: un pequeño cambio en los datos puede cambiar mucho el árbol resultante. Los **ensambles** combinan muchos modelos "débiles" para obtener uno más robusto, de dos formas principales:

- **Bagging** (ej. Random Forest): entrena muchos árboles **en paralelo**, cada uno con una muestra distinta de datos, y promedia sus votos.
- **Boosting** (ej. Gradient Boosting, XGBoost): entrena árboles **en secuencia**, donde cada nuevo árbol intenta corregir los errores del anterior.

<a id="dataset"></a>
## 2. Recuperando el Dataset y Modelos de la Sesión 09

Usamos **el mismo dataset** (`load_breast_cancer`) que en la Sesión 09, para poder comparar directamente los nuevos ensambles contra Logística/KNN/SVM/Árbol.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import pandas as pd

datos = load_breast_cancer(as_frame=True)
X, y = datos.data, datos.target


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)
X_test_esc = escalador.transform(X_test)


In [ ]:
# Los 4 modelos de la Sesión 09, ya entrenados, para benchmarking posterior
modelos_s09 = {
    "Logística": LogisticRegression(max_iter=5000).fit(X_train_esc, y_train),
    "KNN": KNeighborsClassifier(n_neighbors=5).fit(X_train_esc, y_train),
    "SVM": SVC(kernel="rbf", probability=True).fit(X_train_esc, y_train),
    "Árbol": DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train),
}
print("Modelos de la Sesión 09 listos:", list(modelos_s09.keys()))


<a id="rf"></a>
## 3. Random Forest (Bagging)

### 🔬 Teoría técnica
Construye muchos árboles de decisión, cada uno entrenado con una muestra aleatoria de filas (y a veces de columnas), y combina sus votos (clasificación) o promedios (regresión). **Hiperparámetros clave:** `n_estimators` (número de árboles), `max_features`, `max_depth`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)  # los árboles/ensambles de árboles no requieren escalado
print("Accuracy en prueba:", rf.score(X_test, y_test))

### 🧠 Resumen para dummies
"La sabiduría de las masas": muchos árboles ligeramente distintos votan, y el resultado es más estable que confiar en uno solo.

<a id="gb"></a>
## 4. Gradient Boosting

### 🔬 Teoría técnica
En vez de entrenar árboles independientes, entrena uno tras otro **secuencialmente**: cada árbol nuevo se enfoca en corregir los errores (residuos) del conjunto anterior. **Hiperparámetros clave:** `n_estimators`, `learning_rate` (qué tanto "pesa" cada corrección), `max_depth`.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
print("Accuracy en prueba:", gb.score(X_test, y_test))

### 🧠 Resumen para dummies
Cada árbol nuevo es "un alumno que estudia justo lo que el anterior se equivocó" — por eso suele ser más preciso, pero también más lento de entrenar y más sensible a un `learning_rate` mal elegido.

<a id="xgb"></a>
## 5. XGBoost

### 🔬 Teoría técnica
Una implementación de Gradient Boosting muy optimizada (velocidad y regularización integrada), dominante en competencias de datos tabulares. **Hiperparámetros clave:** `n_estimators`, `learning_rate`, `max_depth`, `gamma`, `subsample`.

In [ ]:
# En Colab, xgboost ya viene instalado. Si no, descomenta la siguiente línea:
# !pip install -q xgboost

from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=150, learning_rate=0.1, max_depth=3,
    subsample=0.8, eval_metric="logloss", random_state=42,
)
xgb.fit(X_train, y_train)
print("Accuracy en prueba:", xgb.score(X_test, y_test))

### 💪 Fortalezas y debilidades (resumen)

| Método | Fortaleza | Debilidad |
|---|---|---|
| Random Forest | Robusto, difícil de sobreajustar, fácil de usar | Menos preciso que boosting bien afinado |
| Gradient Boosting | Muy preciso | Sensible al `learning_rate`, más lento |
| XGBoost | Muy preciso, rápido, regularización integrada | Requiere tuning cuidadoso, más hiperparámetros |

### 🧠 Resumen para dummies
Si tienes datos tabulares y quieres el mejor desempeño posible sin mucho esfuerzo, empieza con Random Forest; si necesitas exprimir el último punto porcentual de precisión, ve a XGBoost.

<a id="cv"></a>
## 6. Validación Cruzada k-Fold

### 🔬 Teoría técnica
En vez de confiar en un único `train_test_split` (que puede ser "afortunado" o "desafortunado"), la validación cruzada divide los datos en `k` particiones (folds), entrena `k` veces usando cada una como prueba una sola vez, y reporta el **promedio ± desviación estándar** — una estimación mucho más robusta.

In [ ]:
from sklearn.model_selection import cross_val_score

scores_rf = cross_val_score(rf, X, y, cv=5, scoring="f1")
print(f"Random Forest F1 (5-fold): {scores_rf.mean():.3f} +/- {scores_rf.std():.3f}")

### 🧠 Resumen para dummies
La desviación estándar te dice qué tan "confiable" es el promedio: si es muy alta, el modelo se comporta de forma inconsistente según qué datos le toquen.

<a id="benchmark"></a>
## 7. Benchmarking: la Tabla Comparativa Final

Ahora construimos una función reutilizable que compara **todos** los modelos (Sesión 09 + ensambles de hoy) con validación cruzada, reportando media y desviación estándar.

In [ ]:
from sklearn.pipeline import make_pipeline

def benchmarking(modelos_dict, X, y, cv=5, scoring="f1"):
    filas = []
    for nombre, modelo in modelos_dict.items():
        scores = cross_val_score(modelo, X, y, cv=cv, scoring=scoring)
        filas.append({
            "modelo": nombre,
            f"{scoring}_promedio": scores.mean(),
            f"{scoring}_std": scores.std(),
        })
    return pd.DataFrame(filas).sort_values(by=f"{scoring}_promedio", ascending=False).reset_index(drop=True)


In [ ]:
# Para KNN, SVM y Logística usamos un pipeline con escalado, ya que cross_val_score
# vuelve a hacer fit/predict en cada fold y necesitan estar siempre escalados.
modelos_benchmark = {
    "Logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "SVM": make_pipeline(StandardScaler(), SVC(kernel="rbf")),
    "Árbol": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, max_depth=3, eval_metric="logloss", random_state=42),
}

tabla_benchmark = benchmarking(modelos_benchmark, X, y, cv=5, scoring="f1")
tabla_benchmark


### 🧠 Resumen para dummies
El "modelo campeón" no es solo el de mayor promedio: si dos modelos tienen promedios parecidos pero uno tiene mucha menor desviación estándar, ese es probablemente el más confiable para producción.

## 🔎 Laboratorio de profundización: pérdidas, residuos y boosting

Bagging entrena modelos en paralelo para reducir varianza. Boosting entrena secuencialmente: cada nuevo árbol intenta corregir el error del conjunto.

Para pérdida cuadrática, el gradiente negativo respecto a la predicción coincide con el residuo:

$$-\frac{\partial}{\partial \hat y}\frac{1}{2}(y-\hat y)^2=y-\hat y$$

Por eso Gradient Boosting puede verse como descenso por gradiente en el espacio de funciones:

$$F_m(x)=F_{m-1}(x)+\eta h_m(x)$$


In [ ]:
# Paso 1: inspeccionar hiperparámetros de dos estrategias
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

rf_demo = RandomForestClassifier(
    n_estimators=200, max_depth=None, min_samples_leaf=1,
    max_features="sqrt", random_state=42, n_jobs=-1,
)
gb_demo = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.05, max_depth=2,
    subsample=0.8, random_state=42,
)
print({k: rf_demo.get_params()[k] for k in ["n_estimators", "max_depth", "max_features"]})
print({k: gb_demo.get_params()[k] for k in ["n_estimators", "learning_rate", "subsample"]})


In [ ]:
# Paso 2: validar con los mismos folds para una comparación justa
from sklearn.model_selection import StratifiedKFold, cross_validate

cv_estratificada = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultado_rf = cross_validate(
    rf_demo, X, y, cv=cv_estratificada,
    scoring=["accuracy", "f1", "roc_auc"], n_jobs=-1,
)
pd.DataFrame(resultado_rf).filter(like="test_").agg(["mean", "std"])


### Benchmarking responsable

Compara media **y dispersión**, tiempo de ajuste y una métrica alineada con el costo del error. `n_estimators` controla cantidad de aprendices; `learning_rate` controla el tamaño de cada corrección; `max_depth` controla complejidad; `subsample<1` añade aleatoriedad. No selecciones el campeón usando el conjunto de prueba repetidamente.


<a id="aplicaciones"></a>
## 8. Ejemplos de Aplicación en el Mundo Real

- XGBoost domina las competencias de Kaggle en datos tabulares por su balance entre precisión y velocidad.
- Un equipo de datos rara vez usa "el primer modelo que funcionó": construye una tabla de benchmarking como la de hoy antes de decidir qué llevar a producción.

<a id="retos"></a>
## 9. Retos de Práctica

### 🥉 Reto Básico
Entrena un Random Forest y compara su accuracy (`.score()`) contra el mejor modelo individual de la Sesión 09 (SVM o Logística).

In [ ]:
# Tu solución al Reto Básico aquí


### 🥈 Reto Medio
Usa `cross_val_score` (5-fold) para comparar Random Forest, Gradient Boosting y XGBoost con la métrica `accuracy`, y arma una tabla con media y desviación estándar de cada uno.

In [ ]:
# Tu solución al Reto Medio aquí


### 🥇 Reto Avanzado
Extiende la función `benchmarking` para que reciba también el nombre de la métrica a usar como parámetro, ejecútala con `scoring="roc_auc"` sobre los 7 modelos, y con base en el resultado, justifica en una celda de Markdown cuál es el "modelo campeón" considerando tanto el promedio como la desviación estándar.

In [ ]:
# Tu solución al Reto Avanzado aquí
